# Lattice vertex algebras — examples

Examples for `LatticeVertexAlgebra` (`src/vertex_algebras/lattice_vertex_algebra.py`), a subclass of the common base class `VertexAlgebra`; section 6 combines it with `UniversalVertexAlgebra` in tensor products.

A lattice vertex algebra is given by a coefficient ring, a symmetric, integral, non-degenerate Gram matrix $B$ (not necessarily positive definite) and a cocycle $\varepsilon$ (the number of threads is a setting of the session, `VertexAlgebraParallelism`, not part of the data). The lattice is $\mathbb Z^n$ with basis $x_0,\dots,x_{n-1}$. It is generated by:

* the **Heisenberg currents** $J_i$, with $J_i(z)J_j(w)\sim B_{ij}/(z-w)^2$;
* the **lattice currents** $e^q$ for $q\in\mathbb Z^n$, where $e^0$ is the vacuum.

**Conventions:**

* `a.ope(b)` returns the list of poles: entry $k$ is the coefficient of $(z-w)^{-k-1}$, i.e. $a_{(k)}b$.
* `a.no(b)` is the normal-ordered product $a_{(-1)}b$, and `a.nproduct(b, n)` is $a_{(n)}b$ for any integer $n$.
* $e^q(z)e^p(w)$ starts at $(z-w)^{B(q,p)}$, with $e^q_{(n)}e^p=\varepsilon(q,p)\,S_{-n-1-B(q,p)}(q)\,e^{q+p}$.
* $J_i(z)e^q(w)\sim B(x_i,q)\,e^q(w)/(z-w)$.
* $\partial e^q=\sum_i q_i :J_ie^q:$.
* The parity of $e^q$ is $B(q,q)\bmod 2$.

**No conformal vector is built in.** Every Virasoro current below (Sugawara, or with a background charge) is a choice made in the notebook.

*If the import fails with a stale-module error, restart the kernel and run the import cell first.*

In [1]:
import os, sys
from vertex_algebras import lattice_vertex_algebra
from vertex_algebras.lattice_vertex_algebra import LatticeVertexAlgebra
print('C++ engine loaded:', lattice_vertex_algebra._LVA_ENGINE is not None)

C++ engine loaded: True


In [2]:
def sugawara(L):
    # Tsug = 1/2 sum_ij (B^-1)_ij :J_i J_j:  (central charge = rank)
    Binv = L.gram_matrix().inverse()
    J = L.heisenberg_currents()
    r = L.rank()
    return sum(Binv[i, j] / 2 * J[i].no(J[j]) for i in range(r) for j in range(r))

## 1. The $A_1$ lattice, $B=(2)$: $\widehat{\mathfrak{sl}}_2$ at level 1

$H=J_0$, $E=e^{x_0}$ and $F=e^{-x_0}$ satisfy the $\widehat{\mathfrak{sl}}_2$ OPEs at level 1.

In [3]:
A1 = LatticeVertexAlgebra(QQ, matrix([[2]]))
H = A1.J(0); E = A1.e((1,)); F = A1.e((-1,))
print('H H :', H.ope(H))
print('H E :', H.ope(E), '   H F :', H.ope(F))
print('E F :', E.ope(F))
print('E E :', E.ope(E), '   (regular: B(x,x) = 2 > 0)')
print('first nonzero product of E with E:', E.nproduct(E, -3))

H H : [0, 2]
H E : [2*e^(1)]    H F : [-2*e^(-1)]
E F : [J0, 1]
E E : []    (regular: B(x,x) = 2 > 0)
first nonzero product of E with E: e^(2)


The Sugawara vector gives $c=1$, and $E$, $F$ have conformal weight $B(x,x)/2=1$:

In [4]:
T = sugawara(A1)
print('T      =', T)
print('T T    =', T.ope(T))
print('T E    =', T.ope(E))
print('∂E     =', E.derivative())

T      = 1/4*:J0 J0:
T T    = [1/2*:dJ0 J0:, 1/2*:J0 J0:, 0, 1/2]
T E    = [:J0 e^(1):, e^(1)]
∂E     = :J0 e^(1):


## 2. The cocycle: the $A_2$ lattice

The default **standard** cocycle is bimultiplicative, with $\varepsilon(x_i,x_j)=(-1)^{B_{ij}+B_{ii}B_{jj}}$ for $i>j$ and $1$ otherwise. It satisfies the locality condition $\varepsilon(q,p)/\varepsilon(p,q)=(-1)^{B(q,p)+B(q,q)B(p,p)}$. The trivial cocycle fails this condition for $A_2$, and constructing it raises a warning.

In [5]:
A2 = LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]))
a, b = A2.e((1, 0)), A2.e((0, 1))
print('ε(x0,x1), ε(x1,x0) =', A2.cocycle((1, 0), (0, 1)), A2.cocycle((0, 1), (1, 0)))
print('a b :', a.ope(b), '   b a :', b.ope(a))
print('e^(1,1) e^(-1,-1) :', A2.e((1, 1)).ope(A2.e((-1, -1))))

ε(x0,x1), ε(x1,x0) = 1 -1
a b : [e^(1,1)]    b a : [-e^(1,1)]
e^(1,1) e^(-1,-1) : [-J0 - J1, -1]


In [6]:
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    A2triv = LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]), epsilon='trivial')
print(w[0].message)

LatticeVertexAlgebra: the cocycle violates ε(x_0,x_1)/ε(x_1,x_0) = (-1)^(B_ij + B_ii B_jj); the algebra is not local


## 3. The odd indefinite unimodular lattice $I_{1,1}$, $B=\begin{pmatrix}1&0\\0&-1\end{pmatrix}$

$x_0$ has norm $+1$ and $x_1$ has norm $-1$.

In [7]:
I11 = LatticeVertexAlgebra(QQ, matrix([[1, 0], [0, -1]]))
J0, J1 = I11.heisenberg_currents()
print(I11)
print('ε(x0,x1), ε(x1,x0) =', I11.cocycle((1, 0), (0, 1)), I11.cocycle((0, 1), (1, 0)))
print('J0 J0 :', J0.ope(J0), '  J1 J1 :', J1.ope(J1), '  J0 J1 :', J0.ope(J1))

Lattice vertex algebra of rank 2 with Gram matrix [[1, 0], [0, -1]] over Rational Field
ε(x0,x1), ε(x1,x0) = 1 -1
J0 J0 : [0, 1]   J1 J1 : [0, -1]   J0 J1 : []


### Parities and Sugawara weights

The weight of $e^q$ under $T_{\rm sug}$ is $B(q,q)/2 = (q_0^2-q_1^2)/2$. It can be negative, and it vanishes on the null directions $q_0=\pm q_1$.

In [8]:
Tsug = sugawara(I11)
print('Tsug      =', Tsug)
print('Tsug Tsug =', Tsug.ope(Tsug))
for q in [(1, 0), (0, 1), (1, 1), (1, -1), (2, 1), (1, 2)]:
    eq = I11.e(q)
    print('q = %-8s parity %d   weight %s' % (q, eq.parity(), Tsug.nproduct(eq, 1).coefficient(((), q))))

Tsug      = 1/2*:J0 J0: - 1/2*:J1 J1:
Tsug Tsug = [:dJ0 J0: - :dJ1 J1:, :J0 J0: - :J1 J1:, 0, 1]
q = (1, 0)   parity 1   weight 1/2
q = (0, 1)   parity 1   weight -1/2
q = (1, 1)   parity 0   weight 0
q = (1, -1)  parity 0   weight 0
q = (2, 1)   parity 1   weight 3/2
q = (1, 2)   parity 1   weight -3/2


### Fermions from the norm $+1$ direction: a $bc$ system

$b=e^{x_0}$ and $c=e^{-x_0}$ are odd, with $b(z)c(w)\sim 1/(z-w)$, and $:bc:=J_0$ (fermion–boson correspondence).

In [9]:
b, c = I11.e((1, 0)), I11.e((-1, 0))
print('b c :', b.ope(c), '  c b :', c.ope(b), '  b b :', b.ope(b))
print(':b c: =', b.no(c))

b c : [1]   c b : [1]   b b : []
:b c: = J0


### The norm $-1$ direction

$e^{\pm x_1}$ are also odd. The signs are reversed compared with the norm $+1$ direction: $e^{x_1}(z)e^{-x_1}(w)$ is *regular*, while $e^{x_1}(z)e^{x_1}(w)$ has a *pole*, because $B(x_1,x_1)=-1$.

In [10]:
u, v = I11.e((0, 1)), I11.e((0, -1))
print('u v :', u.ope(v), '  :u v: =', u.no(v))
print('u u :', u.ope(u))
print('v v :', v.ope(v))

u v : []   :u v: = 0
u u : [e^(0,2)]
v v : [e^(0,-2)]


### Null directions

$n_\pm=e^{\pm(x_0+x_1)}$ are even, have $T_{\rm sug}$-weight 0, and $B(n_+,n_-)=0$, so their OPE is regular. Their normal-ordered product is $\varepsilon(x_0+x_1,-x_0-x_1)\,e^0=-1$, a sign from the cocycle.

In [11]:
npl, nmi = I11.e((1, 1)), I11.e((-1, -1))
print('n+ n- :', npl.ope(nmi), '  :n+ n-: =', npl.no(nmi))
print('Tsug n+ :', Tsug.ope(npl))

n+ n- : []   :n+ n-: = -1
Tsug n+ : [:J0 e^(1,1): + :J1 e^(1,1):]


### Background charge

Take $T_\alpha = T_{\rm sug} + a_0\,\partial J_0 + a_1\,\partial J_1$, with symbolic $a_0,a_1$. Use a fraction field, never `var()`. Then:

* the central charge is $c = 2 - 12B(\alpha,\alpha) = 2-12(a_0^2-a_1^2)$;
* the weight of $e^q$ is $B(q,q)/2 - B(\alpha,q)$.

In [12]:
K.<a0, a1> = Frac(QQ['a0', 'a1'])
I11K = LatticeVertexAlgebra(K, matrix([[1, 0], [0, -1]]))
JK = I11K.heisenberg_currents()
Ta = sugawara(I11K) + a0 * JK[0].derivative() + a1 * JK[1].derivative()
TT = Ta.ope(Ta)
print('T T poles 1..3:', TT[:3])
print('c/2 =', TT[3], '   c =', 2 * TT[3].coefficient(((), (0, 0))))
for q in [(1, 0), (0, 1), (1, 1)]:
    print('weight of e^%s = %s' % (q, Ta.nproduct(I11K.e(q), 1).coefficient(((), q))))

T T poles 1..3: [:dJ0 J0: + a0*d^2J0 - :dJ1 J1: + a1*d^2J1, :J0 J0: + 2*a0*dJ0 - :J1 J1: + 2*a1*dJ1, 0]
c/2 = -6*a0^2 + 6*a1^2 + 1    c = -12*a0^2 + 12*a1^2 + 2
weight of e^(1, 0) = -a0 + 1/2
weight of e^(0, 1) = a1 - 1/2
weight of e^(1, 1) = -a0 + a1


### A $\beta\gamma$ system inside $V_{I_{1,1}}$

This is the Friedan–Martinec–Shenker bosonization. Set $\beta = {:}J_0\,e^{x_0-x_1}{:}$ and $\gamma = e^{-x_0+x_1}$. Both are even (the lattice vectors are null), and $\beta(z)\gamma(w)\sim 1/(z-w)$.

Choose the background charge $a_0=\tfrac12$ and $a_1=\tfrac12-\lambda$. Then:

* $\beta$ and $\gamma$ are primary, of weights $\lambda$ and $1-\lambda$;
* $T_\alpha$ **equals** the $\beta\gamma$ stress tensor $\lambda{:}\beta\,\partial\gamma{:} - (1-\lambda){:}\partial\beta\,\gamma{:}$;
* the central charge is $2(6\lambda^2-6\lambda+1)$: $-1$ at $\lambda=\tfrac12$, $2$ at $\lambda=1$, $11$ at $\lambda=\tfrac32$, $26$ at $\lambda=2$.

In [13]:
Kl.<lam> = Frac(QQ['lam'])
V = LatticeVertexAlgebra(Kl, matrix([[1, 0], [0, -1]]))
J0, J1 = V.heisenberg_currents()
beta = V.e((1,1)) #J0.no(V.e((1, -1)))
gamma = J0.no(V.e((-1,-1))) #V.e((-1, 1))
print('parities:', beta.parity(), gamma.parity())
print('β γ :', beta.ope(gamma), '  γ β :', gamma.ope(beta))
print('β β :', beta.ope(beta), '  γ γ :', gamma.ope(gamma))

parities: 0 0
β γ : [1]   γ β : [-1]
β β : []   γ γ : []


In [14]:
T = sugawara(V) - (1/2) * J0.derivative() + (lam-1/2) * J1.derivative()
print('β primary of weight λ   :', T.ope(beta) == [beta.derivative(), (-lam) * beta])
print('γ primary of weight 1-λ :', T.ope(gamma) == [gamma.derivative(), (1+lam) * gamma])
Tbg = lam * beta.no(gamma.derivative()) - (1 - lam) * beta.derivative().no(gamma)
print('T == λ:β∂γ: - (1-λ):∂βγ: :', T == Tbg)
cT = 2 * T.nproduct(T, 3).coefficient(((), (0, 0)))
print('central charge c(λ) =', cT)
print('c at λ = 1/2, 1, 3/2, 2:', [cT.subs(lam=x) for x in (1/2, 1, 3/2, 2)])

β primary of weight λ   : False
γ primary of weight 1-λ : False
T == λ:β∂γ: - (1-λ):∂βγ: : True
central charge c(λ) = 12*lam^2 - 12*lam + 2
c at λ = 1/2, 1, 3/2, 2: [-1, 2, 11, 26]


In [15]:
Tbg.ope(gamma)

[-:J0 J0 e^(-1,-1): - :J0 J1 e^(-1,-1): + :dJ0 e^(-1,-1):,
 (-lam + 1)*:J0 e^(-1,-1):]

In [16]:
T

1/2*:J0 J0: - 1/2*dJ0 - 1/2*:J1 J1: + (lam - 1/2)*dJ1

In [17]:
Tbg

1/2*:J0 J0: - 1/2*dJ0 - 1/2*:J1 J1: + (lam - 1/2)*dJ1

## 4. Checking the axioms by hand

Everything goes through `nproduct`, so identities can be tested directly. Skew-symmetry: $b_{(n)}a = (-1)^{|a||b|}\sum_{j\ge0}(-1)^{n+j+1}\,\partial^{j}\!\left(a_{(n+j)}b\right)/j!$.

In [18]:
def skew_ok(a, b, nrange=range(-3, 3), jmax=8):
    s = (-1) ** (a.parity() * b.parity())
    for n in nrange:
        rhs = sum((-1) ** (n + j + 1) / factorial(j) * a.nproduct(b, n + j).derivative(j)
                  for j in range(jmax))
        if b.nproduct(a, n) != s * rhs:
            return False
    return True

x = I11.J(0).no(I11.e((1, 0)))           # odd
y = I11.e((0, 1)).derivative()           # odd
z = I11.J(1).derivative().no(I11.e((1, 1)))   # even
print(skew_ok(x, y), skew_ok(x, z), skew_ok(y, z))

True True True


## 5. Parallel evaluation and performance

`VertexAlgebraParallelism().set(n)` sets the number of threads of the C++ engines for the whole session (computations the engines do not take are serial); the default is 1 (serial), and results do not depend on it. It plays the role of Sage's `Parallelism()` for this library (Sage's `Parallelism()` has no field for vertex algebras). Coefficient rings are fastest when they are `QQ`, a number field, or `Frac` of a polynomial ring over one of these; `QQbar` is much slower.

In [19]:
import time
from vertex_algebras.vertex_algebra import VertexAlgebraParallelism
VertexAlgebraParallelism().set(4)
print(VertexAlgebraParallelism())
Lp = LatticeVertexAlgebra(QQ, matrix([[1, 0], [0, -1]]))
J = Lp.heisenberg_currents()
X = sum(Lp.e(q) for q in [(1, 0), (0, 1), (1, 1), (-1, 1)]) + J[0].no(J[1]).no(Lp.e((1, -1)))
t = time.time(); poles = X.ope(X.derivative(2)); print('%d poles in %.2fs' % (len(poles), time.time() - t))
print('number of terms per pole:', [len(p.support()) for p in poles])
VertexAlgebraParallelism().reset()   # back to serial

Number of threads for vertex algebra computations: 4
4 poles in 0.01s
number of terms per pole: [39, 20, 11, 6]


## 6. Tensor products

`V.tensor(W)` (or `TensorProductVertexAlgebra([V, W, ...])`) is the super tensor product: $Y(a\otimes b,z)=Y(a,z)\otimes Y(b,z)$ with the Koszul sign $(a\otimes b)_{(n)}(c\otimes d)=(-1)^{|b||c|}\sum_k a_{(k)}c\otimes b_{(n-1-k)}d$. Factors can be any vertex algebras over the same ring. `embed(i, x)` puts `x` into the $i$-th slot, `pure_tensor(x, y, ...)` builds $x\otimes y\otimes\cdots$, and `generators()` collects the generators of all factors.

**$V_{\mathbb Z}\otimes V_{\mathbb Z}\cong V_{I_2}$.** Both factors are fermionic. In the tensor product, elements of different factors anticommute because of the Koszul sign; in the rank-2 lattice VA with $B=\mathrm{diag}(1,1)$ this is done by the cocycle, $\varepsilon(x_1,x_0)=-1$.

In [20]:
from vertex_algebras.tensor_product_vertex_algebra import TensorProductVertexAlgebra
VZ = LatticeVertexAlgebra(QQ, matrix([[1]]))
T = VZ.tensor(VZ)
psi0, psi1 = T.embed(0, VZ.e((1,))), T.embed(1, VZ.e((1,)))
print(T)
print('psi0 psi1 :', psi0.ope(psi1), '   :psi0 psi1: =', psi0.no(psi1), '   :psi1 psi0: =', psi1.no(psi0))
I2 = LatticeVertexAlgebra(QQ, matrix([[1, 0], [0, 1]]))
f0, f1 = I2.e((1, 0)), I2.e((0, 1))
print('in V_I2   :', f0.ope(f1), '   :f0 f1: =', f0.no(f1), '   :f1 f0: =', f1.no(f0))

Tensor product of vertex algebras [Lattice vertex algebra of rank 1 with Gram matrix [1] over Rational Field, Lattice vertex algebra of rank 1 with Gram matrix [1] over Rational Field]
psi0 psi1 : []    :psi0 psi1: = e^(1) # e^(1)    :psi1 psi0: = -e^(1) # e^(1)
in V_I2   : []    :f0 f1: = e^(1,1)    :f1 f0: = -e^(1,1)


**A universal ⊗ lattice product.** The $N=1$ superconformal algebra (a `UniversalVertexAlgebra` with odd $G$) tensored with $V_{I_{1,1}}$. Stress tensors add, $T_{\rm tot}=L\otimes 1+1\otimes T_{\rm sug}$, and so do the central charges ($c_0+2$). The odd fields $G$ and $e^{x_0}$ from different factors anticommute.

In [21]:
from vertex_algebras.universal_vertex_algebra import UniversalVertexAlgebra
from vertex_algebras.normally_ordered_expressions import NormallyOrderedExpressions
K.<c0> = Frac(QQ['c0'])
E.<L, G> = NormallyOrderedExpressions(K, parities={'G': 1})
N1 = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c0/2],
                                   ('L', 'G'): [G.derivative(), 3/2*G],
                                   ('G', 'G'): [2*L, 0, 2*c0/3]})
L, G = N1.gens()
I11K = LatticeVertexAlgebra(K, matrix([[1, 0], [0, -1]]))
M = N1.tensor(I11K)
print('generators:', M.variable_names())
Ttot = M.embed(0, L) + M.embed(1, sugawara(I11K))
TT = Ttot.ope(Ttot)
print('Virasoro:', TT == [Ttot.derivative(), 2*Ttot, 0, M((c0 + 2)/2)], '   c/2 =', TT[3])
g, b = M.embed(0, G), M.embed(1, I11K.e((1, 0)))
print('G, e^x0 parities:', g.parity(), b.parity(), '   :G e^x0: == -:e^x0 G: :', g.no(b) == -b.no(g))
print('T_tot acting on G ⊗ e^x0 :', Ttot.ope(M.pure_tensor(G, I11K.e((1, 0)))))

generators: ('L', 'G', 'J0', 'J1', 'e^(1,0)', 'e^(0,1)', 'e^(-1,0)', 'e^(0,-1)')
Virasoro: True    c/2 = 1/2*c0 + 1
G, e^x0 parities: 1 1    :G e^x0: == -:e^x0 G: : True
T_tot acting on G ⊗ e^x0 : [G # :J0 e^(1,0): + dG # e^(1,0), 2*G # e^(1,0)]


**Heisenberg ⊗ lattice.** The universal free boson $J$ with $J(z)J(w)\sim 1/(z-w)^2$, tensored with $V_{A_1}$, is the subalgebra of $V_{\mathrm{diag}(1,2)}$ with $q_0=0$. It is *not* all of $V_{\mathrm{diag}(1,2)}$: that is $V_{\mathbb Z}\otimes V_{A_1}$.

In [22]:
H = UniversalVertexAlgebra(QQ, ['J'], {('J', 'J'): [0, 1]})
A1 = LatticeVertexAlgebra(QQ, matrix([[2]]))
HA = H.tensor(A1)
Jt = HA.embed(0, H.gen('J'))
Et, Ft = HA.embed(1, A1.e((1,))), HA.embed(1, A1.e((-1,)))
L12 = LatticeVertexAlgebra(QQ, matrix([[1, 0], [0, 2]]))
print('(J ⊗ e^x)(J ⊗ e^-x) :', (Jt * Et).ope(Jt * Ft))
print('in V_diag(1,2)      :', (L12.J(0) * L12.e((0, 1))).ope(L12.J(0) * L12.e((0, -1))))

(J ⊗ e^x)(J ⊗ e^-x) : [1/6*1 # :J0 J0 J0: + 1/2*1 # :dJ0 J0: + 1/6*1 # d^2J0 + :J J: # J0 + :dJ J: # 1, 1/2*1 # :J0 J0: + 1/2*1 # dJ0 + :J J: # 1, 1 # J0, 1]
in V_diag(1,2)      : [:J0 J0 J1: + :dJ0 J0: + 1/6*:J1 J1 J1: + 1/2*:dJ1 J1: + 1/6*d^2J1, :J0 J0: + 1/2*:J1 J1: + 1/2*dJ1, J1, 1]
